In [ ]:
%load_ext blackcellmagic 
# %black -l 120
%load_ext autoreload
%autoreload 2

In [ ]:
from tests.flop_comparison.gidqnshared import GiDQNShared
from tests.flop_comparison.linear_gidqnshared import LinearGiDQNShared
from tests.flop_comparison.idqnshared import iDQNShared
from tests.flop_comparison.linear_idqnshared import LineariDQNShared
from tests.flop_comparison.dqnrcshared import DQNRCShared
from tests.flop_comparison.linear_dqnrcshared import LinearDQNRCShared
from tests.flop_comparison.dqn import DQN

import jax
import jax.numpy as jnp
from tests.utils import Generator


def count_params(params):
	return sum(x.size for x in jax.tree.leaves(params))


def count_flops(q, has_target_params=False):
	best_action_compiled = jax.jit(q.best_action).lower(q.params, sample_generator.state(jax.random.PRNGKey(0))).compile()
	if not has_target_params:
		learn_on_batch_compiled = jax.jit(q.learn_on_batch).lower(q.params, q.optimizer_state, sample_generator.samples(jax.random.PRNGKey(0)), jnp.ones(32)).compile()
	else:
		learn_on_batch_compiled = jax.jit(q.learn_on_batch).lower(q.params, q.target_params, q.optimizer_state, sample_generator.samples(jax.random.PRNGKey(0)), jnp.ones(32)).compile()

	return best_action_compiled, learn_on_batch_compiled

sample_generator = Generator(32, (84, 84, 4), 10) 

architecture = "impala" # "cnn" "impala"
features = [16, 32, 32, 512]  # [32, 64, 64, 512] [16, 32, 32, 512]

print(f"--- Vmapped Heads DQN---")
vmap_q_dqn = DQN(jax.random.PRNGKey(0), (84, 84, 4), 10, features, architecture, False, False, 6.25e-5, 0.99, 1, 1, 8000)
print("DQN with vmap", count_params(vmap_q_dqn.params) + count_params(vmap_q_dqn.target_params))
vmap_q_dqn_best_action_compiled, vmap_q_dqn_learn_on_batch_compiled = count_flops(vmap_q_dqn, has_target_params=True)
print("Vmapped DQN FLOPs best action: ", vmap_q_dqn_best_action_compiled.cost_analysis()[0]["flops"])
print("Vmapped DQN FLOPs to learn on a batch: ", vmap_q_dqn_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")

print(f"--- Vmapped Heads DQNRC---")
vmap_q_dqnrc = DQNRCShared(jax.random.PRNGKey(0), (84, 84, 4), 10, features, architecture, False, False, True, 6.25e-5, 0.99, 1, 1, 8000, 1)
print("DQNRC with vmap", count_params(vmap_q_dqnrc.params))
vmap_q_dqnrc_best_action_compiled, vmap_q_dqnrc_learn_on_batch_compiled = count_flops(vmap_q_dqnrc, has_target_params=False)
print("Vmapped DQNRC FLOPs best action: ", vmap_q_dqnrc_best_action_compiled.cost_analysis()[0]["flops"])
print("Vmapped DQNRC FLOPs to learn on a batch: ", vmap_q_dqnrc_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")


print(f"--- Linear Heads DQNRC ---")
linear_q_dqnrc = LinearDQNRCShared(jax.random.PRNGKey(0), (84, 84, 4), 10, features, architecture, False, False, True, 6.25e-5, 0.99, 1, 1, 8000, 1)
print("DQNRC with linear Heads", count_params(linear_q_dqnrc.params))
linear_q_dqnrc_best_action_compiled, linear_q_dqnrc_learn_on_batch_compiled = count_flops(linear_q_dqnrc, has_target_params=False)
print("Linear DQNRC FLOPs best action: ", linear_q_dqnrc_best_action_compiled.cost_analysis()[0]["flops"])
print("Linear DQNRC FLOPs to learn on a batch: ", linear_q_dqnrc_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")

print(f"--- Vmapped Heads i-DQN---")
vmap_q_idqn = iDQNShared(jax.random.PRNGKey(0), (84, 84, 4), 10, 5, features, architecture, False, False, True, 6.25e-5, 0.99, 1, 1, 8000)
print("iDQN with vmap", count_params(vmap_q_idqn.params) + count_params(vmap_q_idqn.target_params))
vmap_q_idqn_best_action_compiled, vmap_q_idqn_learn_on_batch_compiled = count_flops(vmap_q_idqn, has_target_params=True)
print("Vmapped i-DQN FLOPs best action: ", vmap_q_idqn_best_action_compiled.cost_analysis()[0]["flops"])
print("Vmapped i-DQN FLOPs to learn on a batch: ", vmap_q_idqn_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")


print(f"--- Linear Heads i-DQN ---")
linear_q_idqn = LineariDQNShared(jax.random.PRNGKey(0), (84, 84, 4), 10, 5, features, architecture, False, False, True, 6.25e-5, 0.99, 1, 1, 8000)
print("iDQN with linear Heads", count_params(linear_q_idqn.params) + count_params(linear_q_idqn.target_params))
linear_q_idqn_best_action_compiled, linear_q_idqn_learn_on_batch_compiled = count_flops(linear_q_idqn, has_target_params=True)
print("Linear i-DQN FLOPs best action: ", linear_q_idqn_best_action_compiled.cost_analysis()[0]["flops"])
print("Linear i-DQN FLOPs to learn on a batch: ", linear_q_idqn_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")


print(f"--- Vmapped Heads Gi-DQN---")
vmap_q_gidqn = GiDQNShared(jax.random.PRNGKey(0), (84, 84, 4), 10, 5, features, architecture, False, False, True, 6.25e-5, 0.99, 1, 1, True, 8000, 1)
print("GiDQN with vmap", count_params(vmap_q_gidqn.params) + count_params(vmap_q_gidqn.target_params))
vmap_q_gidqn_best_action_compiled, vmap_q_gidqn_learn_on_batch_compiled = count_flops(vmap_q_gidqn, has_target_params=True)
print("Vmapped FLOPs best action: ", vmap_q_gidqn_best_action_compiled.cost_analysis()[0]["flops"])
print("Vmapped FLOPs to learn on a batch: ", vmap_q_gidqn_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")


print(f"--- Linear Heads Gi-DQN---")
linear_q_gidqn = LinearGiDQNShared(jax.random.PRNGKey(0), (84, 84, 4), 10, 5, features, architecture, False, False, True, 6.25e-5, 0.99, 1, 1, True, 8000, 1)
print("GiDQN with linear Heads", count_params(linear_q_gidqn.params) + count_params(linear_q_gidqn.target_params))
linear_q_gidqn_best_action_compiled, linear_q_gidqn_learn_on_batch_compiled = count_flops(linear_q_gidqn, has_target_params=True)
print("Linear FLOPs best action: ", linear_q_gidqn_best_action_compiled.cost_analysis()[0]["flops"])
print("Linear FLOPs to learn on a batch: ", linear_q_gidqn_learn_on_batch_compiled.cost_analysis()[0]["flops"], "\n")
